# 020 bis — Training: the base architectures on the mockup-free generic split

One last ablation. This notebook is a byte-for-byte copy of the `020_training.ipynb`
training procedure — same architectures, same Round 1 recipe, same
`combined_loss`, same per-architecture subprocess — with **only the dataset
changed**. The standard split (`mockup_aware_train_val_test_split`) does two
things this one deliberately undoes:

| | standard split (`020`) | generic split (this notebook) |
|---|---|---|
| mockup groups (`settings.MOCKUP_ARTWORK_IDS`) | kept, split at pair level into train/val/test | **excluded entirely** — never seen |
| real artworks | grouped: every section of a painting stays in one fold | **ungrouped**: plain pair-level random split |
| folds | train / val / test | **train / val only** (no held-out test) |
| same artwork in val *and* elsewhere | prevented | **allowed** |

The split logic lives in `scripts.dataset.mockup_free_train_val_split`;
`scripts/train_single.py --generic-split` selects it inside each subprocess.
Checkpoints go to a **separate** tree, `models/deterministic_generic/<arch>/`,
so nothing here overwrites the project's real checkpoints.

Evaluation is **not** done here. `data/test/` (the GT-annotated paintings) is
untouched by either split, so it is the common yardstick — see
`C6_mockups_vs_generic.ipynb`, which loads whatever this notebook actually
trained and scores it against `models/deterministic/`.

Make the project root importable so `scripts.*` resolves regardless of the
notebook's working directory.

In [ ]:
import sys
from pathlib import Path

project_root = Path().absolute()
if project_root.name == "notebooks":
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

Imports, a fixed global seed, and a GPU sanity check — identical to `020`
except for the split function imported.

In [ ]:
import matplotlib.pyplot as plt
import tensorflow as tf

from scripts.config import settings
from scripts.dataset import (
    build_dataset,
    load_image_pairs,
    mockup_free_train_val_split,
)
from scripts.reproducibility import set_global_seed
from scripts.visualization import plot_training_curves

set_global_seed()

gpus = tf.config.list_physical_devices("GPU")
print(f"GPUs available: {gpus}")
print(f"TensorFlow version: {tf.__version__}")

## 1. Dataset — mockup-free generic split

Every pair whose artwork ID is in `settings.MOCKUP_ARTWORK_IDS` is dropped.
The remaining real-artwork pairs are shuffled and cut into train / val at the
individual-pair level (`settings.VAL_RATIO`), with no grouping — sections of the
same painting will appear on both sides. There is no test split.

`crop_size` only affects the augmented training split; validation stays
full-image so the `val_loss` the checkpoint selects on is comparable to `020`'s.

In [ ]:
pairs = load_image_pairs(settings.IR_DIR, settings.RGB_DIR)
train_pairs, val_pairs = mockup_free_train_val_split(
    pairs,
    val_ratio=settings.VAL_RATIO,
    mockup_ids=settings.MOCKUP_ARTWORK_IDS,
    seed=settings.SEED,
)

train_ds = build_dataset(
    train_pairs,
    batch_size=settings.BATCH_SIZE,
    augment=True,
    shuffle=True,
    seed=settings.SEED,
    crop_size=settings.CROP_SIZE,
)
val_ds = build_dataset(
    val_pairs,
    batch_size=settings.BATCH_SIZE,
    augment=False,
    shuffle=False,
)

print(f"Train: {len(train_pairs)} patches ({len(train_ds)} batches)")
print(f"Val:   {len(val_pairs)} patches ({len(val_ds)} batches)")
excluded = sorted(settings.MOCKUP_ARTWORK_IDS)
print(f"Excluded mockup IDs: {excluded}")

## 2. Loss function

Unchanged from `020`: the unified `combined_loss` — `0.16 · Charbonnier +
0.84 · (1 − MS-SSIM)` (`fixing.md` #9). Nothing about the loss, optimizer, or
recipe differs here; the only variable under test is the training data.

## 3. Train the architectures

**Select which architectures to train by commenting lines in `ARCHS`.** The
companion `C6_mockups_vs_generic.ipynb` discovers whatever ended up with a
checkpoint under `models/deterministic_generic/` and compares exactly that set —
so a partial run here is fine, C6 adapts.

Each architecture trains in its own **subprocess** (`scripts/train_single.py
--generic-split`), for the same Apple-Silicon GPU-state reason as `020`.
Checkpoints and `history.json` go to `models/deterministic_generic/<arch>/`.
Set `EPOCHS = 2` for a quick smoke test first.

In [ ]:
import json
import subprocess

ARCHS = [
    "resunet",
    "unet",
    "attention_unet",
    "unet_residual",
]
EPOCHS = settings.EPOCHS  # -- lower for a quick smoke test
MODEL_DIR = settings.MODELS_DIR / "deterministic_generic"
LOG_DIR = settings.LOGS_DIR / "deterministic_generic"

histories: dict = {}

for arch in ARCHS:
    cmd = [
        sys.executable,
        "-m",
        "scripts.train_single",
        "--arch",
        arch,
        "--epochs",
        str(EPOCHS),
        "--model-dir",
        str(MODEL_DIR),
        "--log-dir",
        str(LOG_DIR),
        "--generic-split",
    ]
    subprocess.run(cmd, cwd=project_root, check=True)

    history_path = MODEL_DIR / arch / "history.json"
    histories[arch] = json.loads(history_path.read_text())

    best_val_loss = min(histories[arch]["val_loss"])
    print(f"\nBest val_loss ({arch}): {best_val_loss:.4f}")

## 4. Training curves

In [ ]:
for arch, history in histories.items():
    plot_training_curves(history, title=f"Training history — {arch} (generic split)")
    plt.show()

## 5. Summary

Checkpoints saved to `models/deterministic_generic/<arch>/best_model.keras`.
Run `C6_mockups_vs_generic.ipynb` next to score them against the standard-split
checkpoints on `data/test/`.

In [ ]:
for arch in ARCHS:
    ckpt = MODEL_DIR / arch / "best_model.keras"
    status = "found" if ckpt.exists() else "MISSING"
    print(f"{arch:<25}: {status}  ({ckpt})")